In [1]:
from pathlib import Path
import sys

def _find_generators_dir(start: Path) -> Path:
    for p in [start, *start.parents]:
        candidate = p / "src" / "discrete_diffusion" / "data" / "generators"
        if candidate.exists():
            return candidate
    raise RuntimeError("Could not find src/discrete_diffusion/data/generators")

generators_dir = _find_generators_dir(Path.cwd().resolve())
if str(generators_dir) not in sys.path:
    sys.path.insert(0, str(generators_dir))

from brevo import topsort_data, topsort_data_raw


In [2]:
# topsort_data returns tokenized samples: {0: text tokens, 1: token_type metadata, "label": answer-mask}.
# Same graph task in both modes; only node encoding changes.
# multi=False: one graph node is one token id (single-token node encoding).
# Example edge encoding: parent=12, child=5 -> [12, 5]
# Example answer encoding: topo [5, 3, 12] -> [5, 3, 12]
print(topsort_data(N=110))  # Brevo1 (tokenized, single-token node encoding)

# multi=True: one graph node is a short token sequence (multi-token node encoding).
# This is an encoding change, not a different graph/objective.
# Example node-word map: 12->[2,7], 5->[1,6,8], 3->[4,8]
# Example edge encoding: (12->5) -> [2,7, 1,6,8]
# Example answer encoding: [5,3,12] -> [1,6,8, 4,8, 2,7]
print(topsort_data(N=50, multi=True))  # Brevo2 (tokenized, multi-token node encoding)

# Non-tokenized variant: graph structure + query + valid topo answer (no BOS/EOS flattening).
print(topsort_data_raw(N=50))


{0: [9999, 104, 5, 14, 72, 99, 97, 72, 84, 95, 100, 30, 98, 36, 78, 14, 100, 18, 109, 87, 54, 4, 87, 72, 97, 30, 26, 18, 12, 104, 101, 78, 101, 70, 55, 72, 49, 70, 78, 32, 109, 4, 108, 26, 46, 109, 80, 28, 106, 87, 84, 109, 72, 76, 5, 5, 103, 55, 72, 87, 110, 36, 110, 78, 13, 29, 76, 54, 80, 98, 94, 65, 101, 109, 100, 20, 46, 101, 20, 5, 80, 12, 99, 1, 21, 93, 13, 5, 1, 76, 99, 95, 104, 12, 58, 36, 14, 4, 109, 32, 78, 32, 72, 4, 14, 99, 103, 81, 13, 28, 21, 76, 28, 29, 14, 65, 20, 36, 12, 14, 28, 84, 108, 95, 70, 108, 49, 32, 76, 84, 106, 106, 1, 5, 93, 93, 103, 14, 12, 65, 26, 87, 100, 95, 87, 55, 20, 18, 14, 65, 98, 20, 81, 101, 106, 70, 26, 26, 106, 1, 93, 109, 70, 29, 99, 101, 54, 1, 13, 76, 65, 12, 98, 80, 94, 108, 44, 18, 76, 106, 21, 104, 99, 110, 108, 108, 103, 12, 78, 104, 30, 30, 108, 29, 104, 9997, 46, 9998, 18, 36, 4, 29, 14, 32, 95, 12, 109, 70, 76, 65, 78, 55, 104, 101, 20, 30, 26, 46, 9998], 1: [-1, -5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [3]:
# Additional smoke test for the dataset-builder API (without removing existing demos).
def _find_src_dir(start: Path) -> Path:
    for p in [start, *start.parents]:
        candidate = p / "src"
        if (candidate / "discrete_diffusion").exists():
            return candidate
    raise RuntimeError("Could not find repo src directory")

src_dir = _find_src_dir(Path.cwd().resolve())
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from discrete_diffusion.data.datasets import get_brevo_dataset

cfg = {
    "training_samples": 4,
    "evaluation_samples": 3,
    "graph_N": 30,
    "enforce_n_for_training": False,
    "multi_token": False,
}

ds = get_brevo_dataset(cfg)
print(ds)
print("train columns:", ds["train"].column_names)
print("validation columns:", ds["validation"].column_names)
print("train sample prefix:", ds["train"][0]["prefixes"][:180])
print("train sample completion:", ds["train"][0]["completions"][:180])


Generating BREVO validation: 100%|██████████| 3/3 [00:00<00:00, 7315.65it/s]

DatasetDict({
    train: Dataset({
        features: ['prefixes', 'completions'],
        num_rows: 4
    })
    validation: Dataset({
        features: ['prefixes', 'completions'],
        num_rows: 3
    })
})
train columns: ['prefixes', 'completions']
validation columns: ['prefixes', 'completions']
train sample prefix: 9999 1 24 4 18 9 22 1 8 29 19 29 22 8 19 5 29 27 4 4 2 24 8 9 27 27 5 8 5 9 29 1 27 9 8 5 2 24 29 18 3 22 19 4 3 1 9 19 30 29 18 8 27 27 22 22 18 8 18 3 2 24 27 22 3 4 14 24 5 9997
train sample completion: 9998 1 24 9 8 27 5 29 22 19 30 9998


In [4]:
# Loader-path smoke test: build BREVO via get_dataset (tokenization + masks + padding).
from discrete_diffusion.data.loaders import get_dataset
from discrete_diffusion.data.tokenizers import AsciiCharTokenizer

loader_cfg = {
    "training_samples": 8,
    "evaluation_samples": 4,
    "graph_N": 30,
    "enforce_n_for_training": False,
    "multi_token": False,
}

tokenizer = AsciiCharTokenizer()
cache_dir = str(Path.cwd() / ".cache" / "brevo_loader_demo")

train_tok = get_dataset(
    dataset_name="brevo",
    tokenizer=tokenizer,
    wrap=False,
    mode="train",
    cache_dir=cache_dir,
    insert_eos=True,
    insert_special_tokens=True,
    block_size=256,
    num_proc=1,
    streaming=False,
    dataset_config=loader_cfg,
    tokenize=False
)

valid_tok = get_dataset(
    dataset_name="brevo",
    tokenizer=tokenizer,
    wrap=False,
    mode="validation",
    cache_dir=cache_dir,
    insert_eos=True,
    insert_special_tokens=True,
    block_size=256,
    num_proc=1,
    streaming=False,
    dataset_config=loader_cfg,
    tokenize=False
)

print("train rows:", len(train_tok), "valid rows:", len(valid_tok))
print("train columns:", train_tok.column_names)
# Preview using python format to avoid optional torchvision dependency in torch formatter.
example = train_tok.with_format("python")[0]
print("input_ids:", example["input_ids"])
print("attention_mask:", example["attention_mask"])
print("loss_mask:", example["loss_mask"])
print("acc_mask:", example["accuracy_mask"])
print("noise_mask:", example["noise_mask"])

for tok, tid in zip(tokenizer.all_special_tokens, tokenizer.all_special_ids):
    print(f"{tok}: {tid}")

/home/miki/github/PDiff/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Generating BREVO validation: 100%|██████████| 4/4 [00:00<00:00, 7237.80it/s]


Tokenizing:   0%|          | 0/8 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/8 [00:00<?, ? examples/s]

Generating BREVO validation: 100%|██████████| 4/4 [00:00<00:00, 9788.34it/s]


Tokenizing:   0%|          | 0/4 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4 [00:00<?, ? examples/s]

train rows: 8 valid rows: 4
train columns: ['prefixes', 'completions', 'input_ids', 'attention_mask', 'loss_mask', 'accuracy_mask', 'noise_mask']
input_ids: [9999, 5, 3, 25, 8, 28, 15, 14, 18, 12, 6, 26, 8, 17, 4, 28, 7, 2, 10, 2, 5, 12, 28, 14, 6, 10, 25, 11, 14, 25, 23, 6, 27, 4, 15, 5, 17, 4, 24, 28, 9, 14, 5, 11, 10, 23, 7, 14, 7, 17, 29, 5, 19, 17, 27, 2, 12, 17, 23, 11, 5, 10, 20, 26, 14, 12, 21, 25, 6, 19, 3, 23, 6, 19, 21, 8, 28, 12, 8, 26, 25, 6, 21, 10, 27, 8, 3, 23, 3, 26, 10, 30, 14, 30, 25, 25, 18, 18, 15, 2, 23, 18, 4, 11, 20, 21, 15, 20, 29, 30, 12, 19, 29, 30, 5, 20, 19, 9997, 9, 9998, 30, 11, 2, 26, 12, 10, 25, 8, 28, 9, 9998, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 

In [6]:
from discrete_diffusion.data.loaders import get_dataset
from discrete_diffusion.data.tokenizers import BrevoDummyTokenizer

tokenizer = BrevoDummyTokenizer()
print("len:", len(tokenizer))
print("bos/eos/pad/mask:", tokenizer.bos_token_id, tokenizer.eos_token_id, tokenizer.pad_token_id, tokenizer.mask_token_id)

tokenizer.__dict__

len: 10001
bos/eos/pad/mask: 9999 9998 9700 10000


{'vocab_size': 10001,
 'bos_token': '[BOS]',
 'eos_token': '[EOS]',
 'pad_token': '[PAD]',
 'mask_token': '[MASK]',
 'cls_token': '[BOS]',
 'sep_token': '[EOS]',
 'bos_token_id': 9999,
 'eos_token_id': 9998,
 'pad_token_id': 9700,
 'mask_token_id': 10000,
 'padding_side': 'right',
 'truncation_side': 'right'}